In [1]:
import requests
from pprint import pprint
collection_id = 189          
url = "https://api-production.data.gov.sg/v2/public/api/collections/{}/metadata".format(collection_id)
        
response = requests.get(url)
response.raise_for_status()
data = response.json()
child_datasets = data["data"]["collectionMetadata"]["childDatasets"]

print(f"Number of child datasets: {len(child_datasets)}")


Number of child datasets: 5


In [2]:
# -------------------------------------------------------------------------
# Standard column names we want in the final dataset
# -------------------------------------------------------------------------
common_columns = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "resale_price"
]

# -------------------------------------------------------------------------
# Possible variations of column names from the different datasets
# -------------------------------------------------------------------------
column_mapping = {
    "Month": "month",
    "Town": "town",
    "Flat Type": "flat_type",
    "Block": "block",
    "Street Name": "street_name",
    "Storey Range": "storey_range",

    # Different names used for the same field
    "Floor Area": "floor_area_sqm",
    "Floor Area Sqm": "floor_area_sqm",
    "Floor Area sqm": "floor_area_sqm",

    "Flat Model": "flat_model",

    # Different naming variations
    "Lease Commence Date": "lease_commence_date",
    "Lease Commencement Date": "lease_commence_date",

    "Resale Price": "resale_price",

    # Some API responses may already use snake_case
    "month": "month",
    "town": "town",
    "flat_type": "flat_type",
    "block": "block",
    "street_name": "street_name",
    "storey_range": "storey_range",
    "floor_area_sqm": "floor_area_sqm",
    "flat_model": "flat_model",
    "lease_commence_date": "lease_commence_date",
    "resale_price": "resale_price"
}


In [3]:
import pandas as pd
datasets_raw = []
datasets = []
batch_size = 10000

for dataset_id in child_datasets:
    # -------------------------------------------------------------------------
    # 1. Get meta info
    # -------------------------------------------------------------------------
    metaurl = f"https://api-production.data.gov.sg/v2/public/api/datasets/{dataset_id}/metadata"
    metaresponse = requests.get(metaurl)
    metaresponse.raise_for_status()
    dataset_metadata = metaresponse.json()
    dataset_name = dataset_metadata["data"]["name"]

    # -------------------------------------------------------------------------
    # 2. Retrieve actual dataset
    # -------------------------------------------------------------------------
    print(f"\nProcessing: {dataset_id}:\t{dataset_name}")
    datastore_url = "https://data.gov.sg/api/action/datastore_search"

    offset = 0
    dataset_batches = []
    while True:            
        params = {
            "resource_id": dataset_id,
            "limit": batch_size,
            "offset": offset
        }
    
        response = requests.get(
            datastore_url,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        data = response.json()
        records = data["result"]["records"]

        if not records:
            break
        
        batch_df = pd.DataFrame(records)
        dataset_batches.append(batch_df)
        offset += len(batch_df)
        print(f"  Retrieved {offset:,} records", end="\r")

        if len(records) < batch_size:
            break

    if not dataset_batches:
        continue

    dataset_df = pd.concat(dataset_batches, ignore_index=True)
    dataset_df = dataset_df.rename(columns=column_mapping)

    datasets_raw.append(dataset_df.copy())

    dataset_df["source_dataset_id"] = dataset_id
    dataset_df["source_dataset_name"] = dataset_name
    datasets.append(dataset_df)



Processing: d_8b84c4ee58e3cfc0ece0d773c8ca6abc:	Resale flat prices based on registration date from Jan-2017 onwards
  Retrieved 238,677 records
Processing: d_43f493c6c50d54243cc1eab0df142d6a:	Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012
  Retrieved 369,651 records
Processing: d_2d5ff9ea31397b66239f245f57751537:	Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014
  Retrieved 52,203 records
Processing: d_ebc5ab87086db484f88045b47411ebc5:	Resale Flat Prices (Based on Approval Date), 1990 - 1999
  Retrieved 287,196 records
Processing: d_ea9ed51da2787afaf8e51f827c304208:	Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016


In [4]:

# -------------------------------------------------------------------------
# 3. Combine all child datasets
# -------------------------------------------------------------------------
hdb_resale_datasets_raw = pd.concat(datasets_raw, ignore_index=True, sort=False)
hdb_resale_datasets = pd.concat(datasets, ignore_index=True, sort=False)

# Check final shape
print(f"Raw master dataset: {hdb_resale_datasets_raw.shape}")
print(f"Master dataset with source info: {hdb_resale_datasets.shape}")
# Display final columns
print(hdb_resale_datasets.columns.tolist())

Raw master dataset: (984880, 12)
Master dataset with source info: (984880, 14)
['_id', 'month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'source_dataset_id', 'source_dataset_name']


In [5]:
import os
from pathlib import Path
project_path = Path.cwd()

project_path_export = project_path / "export"
project_path_export.mkdir(exist_ok=True)

# -------------------------------------------------------------------------
# Save combined datasets locally
# -------------------------------------------------------------------------

# Raw master dataset containing only source attributes
hdb_resale_datasets_raw.to_csv(project_path_export / "hdb_resale_master_raw.csv", index=False)
# Master dataset containing source tracking information
hdb_resale_datasets.to_csv(project_path_export / "hdb_resale_master_with_source.csv", index=False)
print("Datasets saved successfully.")

# -------------------------------------------------------------------------
# Verify exported files
# -------------------------------------------------------------------------
print(f"Raw dataset: {hdb_resale_datasets_raw.shape[0]:,} rows, \n{hdb_resale_datasets_raw.shape[1]} columns")
print(f"Dataset with source: {hdb_resale_datasets.shape[0]:,} rows, \n{hdb_resale_datasets.shape[1]} columns")

Datasets saved successfully.
Raw dataset: 984,880 rows, 
12 columns
Dataset with source: 984,880 rows, 
14 columns
